In [1]:
import json
import os

# 분석할 파일 경로 리스트
file_paths = ["outputs/kci_records.jsonl", "outputs/kci_records_conf.jsonl"]
unique_categories = set()

for file_path in file_paths:
    if not os.path.exists(file_path):
        print(f"파일을 찾을 수 없습니다: {file_path}")
        continue
        
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                data = json.loads(line.strip())
                category = data.get("categories")
                if category:
                    unique_categories.add(category)
            except json.JSONDecodeError:
                continue

unique_categories_list = sorted(list(unique_categories))
print(f"총 고유 카테고리 수: {len(unique_categories_list)}개")
print("=== 카테고리 목록 ===")
print(unique_categories_list)

파일을 찾을 수 없습니다: outputs/kci_records_conf.jsonl
총 고유 카테고리 수: 137개
=== 카테고리 목록 ===
['가정의학', '가톨릭신학', '간호학', '감성과학', '건축공학', '경영학', '경제학', '고분자공학', '공학', '공학일반', '과학기술학', '관광학', '교육학', '국제/지역개발', '군사학', '금속공학', '기계공학', '기독교신학', '기술정책', '기타공학', '기타동양어문학', '기타사회과학', '기타예술체육', '기타의약학', '기타인문학', '기타자연과학', '내과학', '농수해양학', '농업경제학', '농학', '뇌과학', '독일어와문학', '디자인', '러시아어와문학', '마취과학', '면역학', '무역학', '무용', '문학', '문헌정보학', '물리치료학', '물리학', '미생물학', '미술', '방사선과학', '법학', '병리학', '복합학', '불교학', '비뇨기과학', '사전학', '사회과학', '사회과학일반', '사회복지학', '사회학', '산부인과학', '산업공학', '생물공학', '생물학', '생활과학', '서양고전어와문학', '성형외과학', '소아과학', '수산학', '수의학', '수학', '스페인어와문학', '식품과학', '신경과학', '신경외과학', '신문방송학', '심리과학', '약리학', '약학', '언어학', '여성학', '역사학', '연극', '영어와문학', '영화', '예방의학', '예술일반', '원자력공학', '유교학', '음악학', '응급의학', '의공학', '의약학', '의학일반', '인지과학', '일반외과학', '일본어와문학', '임상병리학', '임학', '자동차공학', '자연과학일반', '자원공학', '작업치료학', '재료공학', '재활의학', '전기공학', '전자/정보통신공학', '정신과학', '정책학', '정치외교학', '정형외과학', '제어계측공학', '조경학', '종교학', '중국어와문학', '지구과학', '지리학', '지역개발', '지역학'

In [6]:
import json
import re
import os
from tqdm import tqdm

target_categories = [
    '가정의학', '간호학', '기타의약학', '내과학', '마취과학', '면역학', 
    '물리치료학', '방사선과학', '병리학', '비뇨기과학', '산부인과학', '성형외과학', 
    '소아과학', '수의학', '신경과학', '신경외과학', '약리학', '약학', '예방의학', 
    '응급의학', '의공학', '의약학', '의학일반', '일반외과학', '임상병리학', 
    '작업치료학', '재활의학', '정신과학', '정형외과학', '치의학', '피부과학', 
    '한의학', '해부학'
]

def analyze_abstract_tokens_by_language(target_category_list):
    # 파일명 내 언더스코어(_)가 유지되도록 정확한 파일명을 배열에 기입합니다.
    file_paths = [
        "outputs/kci_records.jsonl", 
        "outputs/kci_records_conf.jsonl",
        "outputs/kci_records_arti_conf.jsonl"
    ]
    
    valid_file_paths = [fp for fp in file_paths if os.path.exists(fp)]
    
    total_abstracts = 0
    english_only_count = 0
    korean_included_count = 0
    
    # 토큰 누적 변수 분리
    english_word_tokens = 0
    korean_word_tokens = 0
    
    hangul_pattern = re.compile(r'[가-힣]')

    for file_path in valid_file_paths:
        with open(file_path, 'r', encoding='utf-8') as f:
            total_lines = sum(1 for _ in f)
            
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in tqdm(f, total=total_lines, desc=f"{os.path.basename(file_path)} 분석 중"):
                try:
                    data = json.loads(line.strip())
                except json.JSONDecodeError:
                    continue
                    
                category = data.get("categories")
                abstract = data.get("abstract")
                
                if category in target_category_list and abstract:
                    total_abstracts += 1
                    
                    tokens = abstract.split()
                    token_count = len(tokens)
                    
                    # 한글 유무에 따라 토큰 수 분리 합산
                    if not hangul_pattern.search(abstract):
                        english_only_count += 1
                        english_word_tokens += token_count
                    else:
                        korean_included_count += 1
                        korean_word_tokens += token_count
                        
    print("\n[언어별 초록 토큰 개수 비교 결과]")
    print(f"- 분석된 파일: {[os.path.basename(fp) for fp in valid_file_paths]}")
    print(f"- 전체 유효 논문 수: {total_abstracts}개\n")
    
    if english_only_count > 0:
        eng_avg = english_word_tokens / english_only_count
        print(f"[100% 영문 초록]")
        print(f"- 논문 수: {english_only_count}개")
        print(f"- 전체 어절 토큰: {english_word_tokens:,}개")
        print(f"- 평균 어절 토큰: {eng_avg:.2f}개\n")
        
    if korean_included_count > 0:
        kor_avg = korean_word_tokens / korean_included_count
        print(f"[한국어 혼용 초록]")
        print(f"- 논문 수: {korean_included_count}개")
        print(f"- 전체 어절 토큰: {korean_word_tokens:,}개")
        print(f"- 평균 어절 토큰: {kor_avg:.2f}개")

analyze_abstract_tokens_by_language(target_categories)

kci_records_arti_conf.jsonl 분석 중: 100%|███████████████████████████████████| 983635/983635 [00:19<00:00, 51115.31it/s]


[언어별 초록 토큰 개수 비교 결과]
- 분석된 파일: ['kci_records.jsonl', 'kci_records_arti_conf.jsonl']
- 전체 유효 논문 수: 225755개

[100% 영문 초록]
- 논문 수: 170412개
- 전체 어절 토큰: 31,774,493개
- 평균 어절 토큰: 186.46개

[한국어 혼용 초록]
- 논문 수: 55343개
- 전체 어절 토큰: 7,791,142개
- 평균 어절 토큰: 140.78개
